# 01 - Data Audit

## Customer Intelligence Platform

---
- This notebook performs a comprehensive audit of the IBM Telco Customer Churn dataset.  
- We examine the dataset structure, identify data quality issues, assess potential leakage,
and document feature governance decisions.


In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np, pandas as pd

# imports from self defined module
from src.load_data import load_raw
from src.config import LEAKAGE_COLUMNS, GEOGRAPHIC_COLUMNS, TARGET
from src.utils import df_summary


## Dataset Overview

---
We begin by loading the raw dataset and understanding its structure.


In [2]:
df = load_raw()
df_summary(df, "IBM Telco Customer Churn Dataset")


✅ Loaded raw data: 7,043 rows × 33 cols

[INFO] IBM Telco Customer Churn Dataset
   Shape: 7,043 rows x 33 columns

Column Types:
 str        24
int64       6
float64     3
Name: count, dtype: int64

   Memory: 2.93 MB
   Missing values: 5,174


In [3]:
df.head()


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## Column Classification

---
Every column must be classified by its role in the analysis:
- **Target**: The variable we're predicting
- **Feature**: Variables used for prediction
- **Identifier**: Unique keys (excluded from modeling)
- **Leakage**: Variables that encode the target or contain future information
- **Geographic**: Spatial data (loaded separately when needed)


In [4]:
classification = []
for col in df.columns:
    if col == TARGET:
        role = "Target"
    elif col in LEAKAGE_COLUMNS:
        role = "Leakage / Drop"
    elif col in GEOGRAPHIC_COLUMNS:
        role = "Geographic"
    else:
        role = "Feature"

    classification.append({
        "Column": col,
        "Type": str(df[col].dtype),
        "Unique": df[col].nunique(),
        "Missing": df[col].isnull().sum(),
        "Role": role,
    })

class_df = pd.DataFrame(classification)
class_df


,Column,Type,Unique,Missing,Role
0,CustomerID,str,7043,0,Leakage / Drop
1,Count,int64,1,0,Leakage / Drop
2,Country,str,1,0,Leakage / Drop
3,State,str,1,0,Geographic
4,City,str,1129,0,Geographic
5,Zip Code,int64,1652,0,Geographic
6,Lat Long,str,1652,0,Leakage / Drop
7,Latitude,float64,1652,0,Geographic
8,Longitude,float64,1651,0,Geographic
9,Gender,str,2,0,Feature


## Missing Value Assessment

---

In [5]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing Count", ascending=False)
missing_df[missing_df["Missing Count"] > 0]


,Missing Count,Missing %
Churn Reason,5174,73.46


### Findings

---
1. **Churn Reason** has 5,174 missing values (73.5%). This is expected because churn reasons
   are only recorded for customers who have actually churned. This column is classified as leakage.
2. **No other columns** have missing values in their native state.
3. **Total Charges** appears complete but contains blank strings for 11 customers with 0 tenure.
   These will be converted to NaN during cleaning and imputed as $0.


## Leakage Assessment

---
**Critical**: Identifying and removing leakage columns is essential for model validity.  
These columns either directly encode the target or contain information unavailable at prediction time.


In [6]:
leakage_reasons = {
    "CustomerID": "Unique identifier, no predictive value",
    "Count": "Constant value (1 for every row)",
    "Country": "Constant value (United States)",
    "Lat Long": "Duplicate of Latitude/Longitude",
    "Churn Label": "Direct text encoding of target",
    "Churn Score": "Pre-computed churn score (leakage)",
    "CLTV": "IBM-computed CLV (contains future information)",
    "Churn Reason": "Only available after churn occurs",
}

print("Leakage Columns to Drop:")
print("-" * 40)
for col, reason in leakage_reasons.items():
    print(f"  {col}: {reason}")


Leakage Columns to Drop:
----------------------------------------
  CustomerID: Unique identifier, no predictive value
  Count: Constant value (1 for every row)
  Country: Constant value (United States)
  Lat Long: Duplicate of Latitude/Longitude
  Churn Label: Direct text encoding of target
  Churn Score: Pre-computed churn score (leakage)
  CLTV: IBM-computed CLV (contains future information)
  Churn Reason: Only available after churn occurs


## Target Variable Analysis

---

In [7]:
print(f"Target Variable: {TARGET}")
print(f"\nValue Counts:")
print(df[TARGET].value_counts())
print(f"\nChurn Rate: {df[TARGET].mean():.2%}")
print(f"Class Ratio: {df[TARGET].value_counts()[0] / df[TARGET].value_counts()[1]:.2f}:1")


Target Variable: Churn Value

Value Counts:
Churn Value
0    5174
1    1869
Name: count, dtype: int64

Churn Rate: 26.54%
Class Ratio: 2.77:1


### Interpretation

---
The dataset contains **7,043 customers** with a churn rate of **26.54%** (approximately 1 in 4 customers).  
This represents moderate class imbalance (2.77:1 ratio), which is manageable with standard techniques like class weighting or threshold optimization.

---

## Summary

| Item | Value |
|------|-------|
| Total Records | 7,043 |
| Total Features | 33 |
| Target | Churn Value (binary 0/1) |
| Churn Rate | 26.54% |
| Leakage Columns | 8 (will be dropped) |
| Geographic Columns | 5 (loaded separately) |
| Missing Values | Churn Reason (73.5%), Total Charges (11 blank strings) |
